In [5]:
# ==========================
# Classificação de Lentes de Contato
# ==========================

import pandas as pd

# --------------------------
# 1. Dataset
# --------------------------
data = {
    "idade": ["infantil","infantil","infantil","infantil",
              "adolescente","adolescente","adolescente","adolescente","adolescente",
              "adulto","adulto","adulto","adulto","adulto","adulto"],

    "diagnostico": ["miopia","miopia","hipermetropia","hipermetropia",
                    "miopia","miopia","miopia","hipermetropia","hipermetropia",
                    "miopia","miopia","miopia","hipermetropia","hipermetropia","hipermetropia"],

    "astigmatismo": ["nao","sim","nao","sim",
                     "nao","sim","nao","nao","sim",
                     "nao","sim","sim","nao","sim","nao"],

    "lacrimal": ["reduzida","normal","normal","normal",
                 "reduzida","reduzida","normal","reduzida","normal",
                 "normal","normal","normal","reduzida","normal","normal"],

    "lente": ["nenhuma","gelatinosa","gelatinosa","dura",
              "gelatinosa","nenhuma","dura","gelatinosa","dura",
              "gelatinosa","dura","gelatinosa","nenhuma","gelatinosa","gelatinosa"]
}

df = pd.DataFrame(data)

# --------------------------
# 2. Análise exploratória
# --------------------------
print("\n=== INFO ===")
df.info()

print("\n=== DESCRIBE ===")
print(df.describe(include='all'))

# --------------------------
# 3. Tabelas de contagem
# --------------------------
print("\n=== TABELAS DE CONTAGEM ===")
for atributo in ["idade", "diagnostico", "astigmatismo", "lacrimal"]:
    print(f"\n{atributo} vs lente:")
    print(pd.crosstab(df[atributo], df["lente"]))

# --------------------------
# 4. Naive Bayes (Laplace)
# --------------------------
alpha = 1
classes = df["lente"].unique()
atributos = ["idade", "diagnostico", "astigmatismo", "lacrimal"]

cond_prob = {}

for atributo in atributos:
    cond_prob[atributo] = {}
    for classe in classes:
        subset = df[df["lente"] == classe]
        total = len(subset)
        valores = df[atributo].unique()
        k = len(valores)

        probs = {}
        for v in valores:
            count = len(subset[subset[atributo] == v])
            probs[v] = (count + alpha) / (total + alpha * k)

        cond_prob[atributo][classe] = probs

# --------------------------
# 5. Paciente
# --------------------------
paciente = {
    "idade": "infantil",
    "diagnostico": "hipermetropia",
    "astigmatismo": "nao",
    "lacrimal": "reduzida"
}

# --------------------------
# 6. Classificação Naive Bayes
# --------------------------
prior = df["lente"].value_counts(normalize=True)

resultados = {}

for classe in classes:
    prob = prior[classe]
    for atributo in atributos:
        prob *= cond_prob[atributo][classe][paciente[atributo]]
    resultados[classe] = prob

# Normalizar
soma = sum(resultados.values())
for c in resultados:
    resultados[c] /= soma

print("\n=== NAIVE BAYES ===")
print("Probabilidades:", resultados)
print("Classe prevista:", max(resultados, key=resultados.get))

# --------------------------
# 7. Rede Bayesiana
# lacrimal depende de idade
# --------------------------
cond_lacrimal = {}

for classe in classes:
    cond_lacrimal[classe] = {}
    for idade in df["idade"].unique():
        subset = df[(df["lente"] == classe) & (df["idade"] == idade)]
        total = len(subset)

        valores = df["lacrimal"].unique()
        k = len(valores)

        probs = {}
        for v in valores:
            count = len(subset[subset["lacrimal"] == v])
            probs[v] = (count + 1) / (total + k)

        cond_lacrimal[classe][idade] = probs

# Classificação com Rede Bayesiana
resultados_rede = {}

for classe in classes:
    prob = prior[classe]

    # independentes
    for atributo in ["idade", "diagnostico", "astigmatismo"]:
        prob *= cond_prob[atributo][classe][paciente[atributo]]

    # dependência
    prob *= cond_lacrimal[classe][paciente["idade"]][paciente["lacrimal"]]

    resultados_rede[classe] = prob

# Normalizar
soma = sum(resultados_rede.values())
for c in resultados_rede:
    resultados_rede[c] /= soma

print("\n=== REDE BAYESIANA ===")
print("Probabilidades:", resultados_rede)
print("Classe prevista:", max(resultados_rede, key=resultados_rede.get))


=== INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   idade         15 non-null     object
 1   diagnostico   15 non-null     object
 2   astigmatismo  15 non-null     object
 3   lacrimal      15 non-null     object
 4   lente         15 non-null     object
dtypes: object(5)
memory usage: 732.0+ bytes

=== DESCRIBE ===
         idade diagnostico astigmatismo lacrimal       lente
count       15          15           15       15          15
unique       3           2            2        2           3
top     adulto      miopia          nao   normal  gelatinosa
freq         6           8            8       10           8

=== TABELAS DE CONTAGEM ===

idade vs lente:
lente        dura  gelatinosa  nenhuma
idade                                 
adolescente     2           2        1
adulto          1           4        1
infantil        1   